# E1.11 · Model risk management for AI systems

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.10 · The stakeholder map: who owns what](https://spbreed.github.io/cyber-commons/lessons/E1.10.html)**.

| | |
|---|---|
| Tools used | Inspect |

## What this lesson is

**What it covers.** Take a validated model, add one tool, and show which parts of the validation are now void.

**Why a security engineer needs it.** The classical model-risk playbook silently breaks once the model can act: conceptual soundness was validated, and then the agent was granted write access nobody validated. The control it builds is: extend the SR 11-7 lineage — conceptual soundness, ongoing monitoring, independent validation — to non-deterministic, tool-using systems, and name where it still holds.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Model risk management has forty years of doctrine on validating models — conceptual soundness, ongoing monitoring, independent validation. Most of it transfers. The part that does not is the part where the model calls tools.

> **At CyberTravels.** Forty years of model-risk doctrine transfers to CyberTravels. The part that does not is the part where the model calls `issue_refund`.

## 2 · The framework

```
   SR 11-7 lineage                 what an agent adds
   +----------------------+        +-----------------------+
   | conceptual soundness |  ok    | it calls tools        |
   | ongoing monitoring   |  ok    | it acts on the world  |
   | independent validation| ok    | output is not a number|
   +----------------------+        +-----------------------+

   most of forty years of doctrine transfers. the tool call does not.
```

Model risk management is not new. The SR 11-7 lineage has governed models in
regulated institutions for over a decade, and its three pillars are sound:

1. **Conceptual soundness** — is the method appropriate for the purpose?
2. **Ongoing monitoring** — is it still performing as validated?
3. **Independent validation** — did someone other than the builder check?

All three still hold for AI systems. What breaks is not the framework but a
silent assumption underneath it: **that a model produces an output, and a human
decides what to do with it.**

Once the model can call a tool, that assumption is void. Validation scoped to
the model's *predictions* says nothing about the model's *actions*. You can hold
a perfectly valid validation report for a system that has since been granted
write access to a production database, and nothing in the classical process is
required to notice.

So the extension is narrow and specific: the unit of validation becomes the
**model plus its tool surface plus its autonomy level**, and any change to any of
the three triggers revalidation — not just a change to the weights.

## 3 · The procedure, as a skill

A model validated with no tools at L1 is deployed with three tools at L3 — same model, same version, different system. The skill diffs validated against deployed and lists what the monitoring never observes.

In [ ]:
# skills/grc/model-risk-validation-scope/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: model-risk-validation-scope
description: >-
  Check whether a model validated under one autonomy level and tool set is still
  covered by that validation as deployed, and find the monitoring that stops at
  the model's output. Use when applying model risk management to an agentic
  system.
allowed-tools: Read, Grep, Glob
---

# The validation was of a system that had no tools

Model risk management carries three pillars — validation, monitoring,
governance — and each makes an assumption that agentic deployment breaks.
Validation assumes the thing validated is the thing deployed; a model validated
with no tools at L1 and deployed with three tools at L3 is the same model and a
different system, and the validation does not cover it.

## When to use this

Applying an existing model risk framework to agents, and at every autonomy or
tool-manifest change afterwards.

## Procedure

**1 — Write down what was validated.** Model, version, tool set, autonomy level,
data scope. Validation scope is usually recorded as the model alone, which is
the defect.

**2 — Write down what is deployed,** in the same fields. Then diff. Any
difference in tools or autonomy means the validation does not cover the
deployment, and that sentence is the finding.

**3 — Check what monitoring observes.** Most monitors the model's output
distribution. List what it does not observe — rows written to production, actions
taken, resources reached — because that is where agentic risk lives.

**4 — Name the re-validation triggers.** Tool added, autonomy raised, model
version changed, data scope widened. Without triggers, re-validation happens on
the audit calendar, which is the assumption that failed in the first place.

**5 — Report per pillar.** Validation coverage, monitoring blind spots,
governance triggers. Three findings with three owners rather than one finding
about the framework.

## Output contract

```json
{
  "validated": {"model": "str", "version": "str", "tools": ["str"], "autonomy": "str", "data": ["str"]},
  "deployed": {"model": "str", "version": "str", "tools": ["str"], "autonomy": "str", "data": ["str"]},
  "covers": false,
  "differences": ["str"],
  "monitoring": {"observes": ["str"], "does_not_observe": ["str"]},
  "revalidation_triggers": ["str"]
}
```

## Failure modes

- **Recording validation scope as the model.** It is the system.
- **Monitoring output distribution only.** The actions are the risk.
- **Calendar re-validation.** The triggers are events, not dates.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/grc/model-risk-validation-scope/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/grc/model-risk-validation-scope/scripts/model_risk_validation_scope.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Check whether a model validated at one autonomy level and tool set is still covered by that validation as deployed.

This is the executable half of the `model-risk-validation-scope` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

PILLARS = {
 "conceptual_soundness": ("is the method appropriate for the purpose",
                          "assumes the purpose is stable and stated"),
 "ongoing_monitoring":   ("is it still performing as validated",
                          "assumes performance is what changes"),
 "independent_validation":("did someone other than the builder check",
                          "assumes the thing checked is the thing deployed"),
}
print(f"{'pillar':24s}{'what it asks':46s}what it quietly assumes")
for p in sorted(PILLARS):
    asks, assumes = PILLARS[p]
    print(f"{p:24s}{asks:46s}{assumes}")
print()
print("All three survive contact with AI. The assumptions are what break.")

VALIDATED = {
 "model": "glm-5.2", "version": "2026-03",
 "purpose": "summarise support tickets",
 "tools": [],                       # at validation time it had none
 "autonomy": "L1",                  # suggests; a human acts
}

def validation_covers(deployed, validated):
    diffs = []
    if deployed["model"] != validated["model"]:       diffs.append("model changed")
    if deployed["version"] != validated["version"]:   diffs.append("version changed")
    if deployed["purpose"] != validated["purpose"]:   diffs.append("purpose changed")
    if sorted(deployed["tools"]) != sorted(validated["tools"]):
        diffs.append(f"tool surface changed: {sorted(set(deployed['tools']) - set(validated['tools']))}")
    if deployed["autonomy"] != validated["autonomy"]: diffs.append(
        f"autonomy raised {validated['autonomy']} -> {deployed['autonomy']}")
    return (not diffs), diffs

DEPLOYED = dict(VALIDATED, tools=["read_ticket", "write_ticket", "db_update"],
                autonomy="L3")
ok, diffs = validation_covers(DEPLOYED, VALIDATED)
print(f"validation still covers what is deployed: {ok}")
for d in diffs:
    print(f"   {d}")
print()
print("Same weights. Same version. The validation report is accurate about a")
print("system that no longer exists, and nothing in the classical process is")
print("required to notice, because the classical trigger is a model change.")
assert not ok

import random
def monitor(metric, runs=200, seed=4):
    rng = random.Random(seed)
    return [round(rng.gauss(0.92, 0.01), 3) for _ in range(runs)]

acc = monitor("summarisation_accuracy")
print(f"summarisation accuracy over 200 runs: mean {sum(acc)/len(acc):.3f}, "
      f"min {min(acc)}, max {max(acc)}")
print("threshold 0.85 -> breaches:", sum(a < 0.85 for a in acc))
print()
UNMONITORED = ["rows written to production", "tools invoked per run",
               "actions taken without human review", "scope of the credential used"]
print("what is NOT on the dashboard:")
for u in UNMONITORED:
    print(f"   {u}")
print()
print("The monitoring is excellent and it is monitoring the prediction. The")
print("risk moved to the action, and the action has no threshold, no baseline")
print("and no alert.")
assert sum(a < 0.85 for a in acc) == 0

TRIGGERS = {
 "model or version change": True,
 "prompt or config change": True,
 "tool added or scope widened": True,
 "autonomy level raised": True,
 "purpose changed": True,
 "calendar year elapsed": True,
}
CLASSICAL = {"model or version change", "calendar year elapsed"}

print(f"{'trigger':32s}{'classical MRM':16s}extended")
for t in TRIGGERS:
    print(f"{t:32s}{'yes' if t in CLASSICAL else 'no':16s}yes")
missed = [t for t in TRIGGERS if t not in CLASSICAL]
print(f"\ntriggers classical MRM would miss: {len(missed)}")
for m in missed: print(f"   {m}")
print()
ok2, diffs2 = validation_covers(DEPLOYED, VALIDATED)
print(f"under the extended triggers, this deployment requires revalidation: {not ok2}")
print(f"reasons: {diffs2}")
assert len(missed) == 4

record = {
 "model": DEPLOYED["model"], "version": DEPLOYED["version"],
 "purpose": DEPLOYED["purpose"],
 "tool_surface": sorted(DEPLOYED["tools"]),
 "autonomy": DEPLOYED["autonomy"],
 "validated_unit": "model + tool surface + autonomy",
 "monitors": ["summarisation_accuracy", "rows_written", "tools_per_run",
              "actions_without_review"],
 "revalidation_triggers": sorted(TRIGGERS),
 "independent_of_builder": True,
}
for k in sorted(record):
    print(f"   {k:24s}{record[k]}")
print()
print("Three fields carry the whole extension: validated_unit, tool_surface and")
print("autonomy. Without them a validation report describes a text generator,")
print("and the thing in production is an actor.")
assert record["validated_unit"].startswith("model + tool")

## What you just proved

The three SR 11-7 pillars, each with the assumption it quietly makes. A system validated with no tools at L1 is shown deployed with three tools at L3 — same model, same version — and the validation no longer covers it. Monitoring reports 200 clean runs of summarisation accuracy while four action-level metrics have no threshold at all, and four revalidation triggers classical MRM would miss are named.

## Your turn

Take one validated model in your estate and list the tools it holds today. If any of them post-dates the validation report, the report is describing a different system.

---

**Next → [E1.12 · Working the seams](https://spbreed.github.io/cyber-commons/lessons/E1.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*